<a href="https://colab.research.google.com/github/dsaenzd04/IA_PROGRAMACION/blob/main/modelo_bigramas_didactico_corregido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
def limpiar_texto(texto, caracteres_a_quitar=None):
    """Pasa a minusculas y elimina puntuacion basica."""
    if caracteres_a_quitar is None:
        caracteres_a_quitar = ['.', ',', '!', '¿', '?', '¡', '(', ')', '"']
    texto_limpio = texto.lower()
    for caracter in caracteres_a_quitar:
        texto_limpio = texto_limpio.replace(caracter, '')
    return texto_limpio


def tokenizar(texto_limpio):
    """Divide el texto limpio en una lista de palabras."""
    return texto_limpio.split()


def construir_vocabulario_codificar(palabras, token_unk='<UNK>'):
    """Crea el diccionario palabra -> ID. El ID 0 es para <UNK>."""
    vocab = {token_unk: 0}
    for palabra in palabras:
        if palabra not in vocab:
            vocab[palabra] = len(vocab)
    return vocab


def construir_vocabulario_decodificar(vocab_codificar):
    """Invierte el vocabulario para obtener ID -> palabra."""
    return {id_num: palabra for palabra, id_num in vocab_codificar.items()}


def codificar_frase(frase_nueva, vocab_codificar):
    """Convierte una frase en IDs. Palabras desconocidas -> 0 (<UNK>)."""
    palabras = tokenizar(limpiar_texto(frase_nueva))
    return [vocab_codificar.get(p, 0) for p in palabras]


def decodificar_id(codigo, vocab_decodificar):
    """Devuelve la palabra de un ID o <UNK> si no existe."""
    return vocab_decodificar.get(codigo, '<UNK>')


def decodificar_secuencia(secuencia_numerica, vocab_decodificar):
    """Convierte una lista de IDs en una cadena de texto."""
    palabras = [decodificar_id(c, vocab_decodificar) for c in secuencia_numerica]
    return ' '.join(palabras)


def mostrar_vocab_aprendido(vocab):
    """Imprime el vocabulario de forma legible."""
    for clave, valor in vocab.items():
        print(f"{clave} : {valor}")


print("Funciones base cargadas correctamente.")

Funciones base cargadas correctamente.


In [11]:
def preparar_corpus(texto_crudo):
    """Devuelve una lista de oraciones, cada una como lista de tokens limpios."""
    oraciones = []
    for linea in texto_crudo.strip().split('\n'):
        limpia = limpiar_texto(linea)
        tokens = tokenizar(limpia)
        if tokens:                      # ignoramos lineas vacias
            oraciones.append(tokens)
    return oraciones


def recolectar_vocabulario(oraciones):
    """Construye el vocabulario (palabra -> ID) a partir de una lista de oraciones."""
    todas_las_palabras = [p for oracion in oraciones for p in oracion]
    return construir_vocabulario_codificar(todas_las_palabras)


# --- Prueba rapida ---
corpus_demo = """La Inteligencia Artificial es el futuro de la tecnologia.
Los diccionarios en Python son extremadamente rapidos.
Las tablas hash permiten busquedas en tiempo constante."""

oraciones = preparar_corpus(corpus_demo)
vocab_cod = recolectar_vocabulario(oraciones)
vocab_dec = construir_vocabulario_decodificar(vocab_cod)

print("Numero de oraciones:", len(oraciones))
for i, o in enumerate(oraciones, 1):
    print(f"   Oracion {i}: {o}")

print("\nTamano del vocabulario:", len(vocab_cod))
print("Palabras del vocabulario:")
mostrar_vocab_aprendido(vocab_cod)

Numero de oraciones: 3
   Oracion 1: ['la', 'inteligencia', 'artificial', 'es', 'el', 'futuro', 'de', 'la', 'tecnologia']
   Oracion 2: ['los', 'diccionarios', 'en', 'python', 'son', 'extremadamente', 'rapidos']
   Oracion 3: ['las', 'tablas', 'hash', 'permiten', 'busquedas', 'en', 'tiempo', 'constante']

Tamano del vocabulario: 23
Palabras del vocabulario:
<UNK> : 0
la : 1
inteligencia : 2
artificial : 3
es : 4
el : 5
futuro : 6
de : 7
tecnologia : 8
los : 9
diccionarios : 10
en : 11
python : 12
son : 13
extremadamente : 14
rapidos : 15
las : 16
tablas : 17
hash : 18
permiten : 19
busquedas : 20
tiempo : 21
constante : 22


In [12]:
def contar_trigramas(oraciones):
    """Devuelve un dict con los triples (w1, w2, w3) y su frecuencia."""
    trigramas = {}
    for tokens in oraciones:
        for i in range(len(tokens) - 2):
            par = (tokens[i], tokens[i + 1],tokens[i+2])
            trigramas[par] = trigramas.get(par, 0) + 1
    return trigramas


def contar_contextos(oraciones):
    """Cuenta cuantas veces aparece cada palabra como primera de un bigrama."""
    contextos = {}
    for tokens in oraciones:
        for i in range(len(tokens) - 2):
            palabra = tokens[i]
            contextos[palabra] = contextos.get(palabra, 0) + 2
    return contextos


def top_n(diccionario, n):
    """Devuelve los n triples (clave, valor) mas frecuentes."""
    return sorted(diccionario.items(), key=lambda kv: kv[1], reverse=True)[:n]


# --- Prueba rapida ---
trigramas = contar_trigramas(oraciones)
contextos = contar_contextos(oraciones)

print("Trigramas mas frecuentes:")
for triple, freq in top_n(trigramas, 5):
    print(f"   {triple} -> {freq}")

print("\nContextos mas frecuentes (denominadores de la probabilidad):")
for palabra, freq in top_n(contextos, 5):
    print(f"   {palabra} -> {freq}")

print("\nComprobacion manual:")
print(f"   El bigrama ('el', 'futuro','de') aparece {trigramas.get(('el','futuro','de'), 0)} vez/veces")
print(f"   La palabra 'el' es contexto {contextos.get('el', 0)} vez/veces")
print(f"   Por tanto P('de'|'futuro' | 'el') = {trigramas.get(('el','futuro','de'),0)} / {contextos.get('el',0)}")

Trigramas mas frecuentes:
   ('la', 'inteligencia', 'artificial') -> 1
   ('inteligencia', 'artificial', 'es') -> 1
   ('artificial', 'es', 'el') -> 1
   ('es', 'el', 'futuro') -> 1
   ('el', 'futuro', 'de') -> 1

Contextos mas frecuentes (denominadores de la probabilidad):
   en -> 4
   la -> 2
   inteligencia -> 2
   artificial -> 2
   es -> 2

Comprobacion manual:
   El bigrama ('el', 'futuro','de') aparece 1 vez/veces
   La palabra 'el' es contexto 2 vez/veces
   Por tanto P('de'|'futuro' | 'el') = 1 / 2


In [13]:
def probabilidad_trigrama(w1, w2,w3, trigramas, contextos, vocab_size):
    """P(w2 | w1) con suavizado de Laplace."""
    conteo_par = trigramas.get((w1, w2,w3), 0)
    conteo_contexto = contextos.get((w1,w2), 0)
    return (conteo_par + 1) / (conteo_contexto + vocab_size)


def distribucion_siguiente(w1,w2, trigramas, contextos, vocab_decodificar):
    """Devuelve un dict {palabra: probabilidad} de TODAS las palabras del vocabulario."""
    vocab_size = len(vocab_decodificar)
    distribucion = {}
    for palabra in vocab_decodificar.values():
        distribucion[palabra] = probabilidad_trigrama(w1,w2, palabra, trigramas, contextos, vocab_size)
    return distribucion


# --- Prueba rapida ---
V = len(vocab_cod)
print("Tamano del vocabulario V =", V)

print("\nProbabilidades de algunas continuaciones de 'el':")
for w in ['futuro', 'python', 'tecnologia', 'hash']:
    p = probabilidad_trigrama('el',"futuro", w, trigramas, contextos, V)
    print(f"   P('{w}' | 'el') = {p:.4f}")

dist = distribucion_siguiente('el',"de", trigramas, contextos, vocab_dec)
print("\nTop 5 continuaciones de 'el':")
for palabra, p in top_n(dist, 5):
    print(f"   {palabra:15s} {p:.4f}")

suma = sum(dist.values())
print(f"\nSuma de todas las probabilidades: {suma:.6f}  (debe ser = 1)")
print(f"La distribucion contiene <UNK>? {'<UNK>' in dist}")

Tamano del vocabulario V = 23

Probabilidades de algunas continuaciones de 'el':
   P('futuro' | 'el') = 0.0435
   P('python' | 'el') = 0.0435
   P('tecnologia' | 'el') = 0.0435
   P('hash' | 'el') = 0.0435

Top 5 continuaciones de 'el':
   <UNK>           0.0435
   la              0.0435
   inteligencia    0.0435
   artificial      0.0435
   es              0.0435

Suma de todas las probabilidades: 1.000000  (debe ser = 1)
La distribucion contiene <UNK>? True


In [14]:
import random

def muestrear_siguiente_palabra(w1,w2, trigramas, contextos, vocab_decodificar):
    """Dada una palabra, devuelve la siguiente muestreada segun P(· | w1),
    excluyendo <UNK> de las candidatas."""
    distribucion = distribucion_siguiente(w1,w2, trigramas, contextos, vocab_decodificar)
    # Filtramos <UNK>: no queremos que aparezca en texto generado
    candidatas = {p: pr for p, pr in distribucion.items() if p != '<UNK>'}
    palabras = list(candidatas.keys())
    pesos    = list(candidatas.values())
    return random.choices(palabras, weights=pesos, k=1)[0]


# --- Prueba rapida ---
random.seed(42)
print("10 muestras de P(· | 'el'):")
for _ in range(10):
    print("   ", muestrear_siguiente_palabra('el',"futuro", trigramas, contextos, vocab_dec))

# Comprobacion estadistica: coincide con lo teorico?
random.seed(42)
N = 5000
conteos = {}
for _ in range(N):
    w = muestrear_siguiente_palabra('el',"futuro", trigramas, contextos, vocab_dec)
    conteos[w] = conteos.get(w, 0) + 1

print(f"\nFrecuencia empirica vs teorica ({N} muestras):")
print(f"   {'palabra':15s} {'empirica':>10s} {'teorica':>10s}")
for palabra, freq in top_n(conteos, 5):
    empirica = freq / N
    teorica = probabilidad_trigrama('el',"futuro", palabra, trigramas, contextos, V)
    print(f"   {palabra:15s} {empirica:10.4f} {teorica:10.4f}")

print("\nCuantas mas muestras, mas se parecen empirica y teorica (ley de los grandes numeros).")

10 muestras de P(· | 'el'):
    extremadamente
    la
    de
    futuro
    las
    rapidos
    busquedas
    inteligencia
    los
    la

Frecuencia empirica vs teorica (5000 muestras):
   palabra           empirica    teorica
   de                  0.0920     0.0870
   python              0.0486     0.0435
   permiten            0.0474     0.0435
   rapidos             0.0472     0.0435
   extremadamente      0.0462     0.0435

Cuantas mas muestras, mas se parecen empirica y teorica (ley de los grandes numeros).


In [15]:
def generar_frase(semilla, trigramas, contextos, vocab_decodificar, n_palabras=10):
    """Genera una frase de n_palabras a partir de una semilla."""
    frase = [semilla]
    actual = semilla
    for _ in range(n_palabras - 1):
        actual = muestrear_siguiente_palabra(actual, trigramas, contextos, vocab_decodificar)
        frase.append(actual)
    return frase


# --- Prueba rapida ---
random.seed(42)
print("Tres frases generadas desde 'el':")
for i in range(3):
    frase = generar_frase('el', trigramas, contextos, vocab_dec, n_palabras=10)
    print(f"   Frase {i+1}: {' '.join(frase)}")

print("\nFrases desde distintas semillas:")
random.seed(1)
for s in ['la', 'los', 'el', 'python', 'hash', 'inteligencia']:
    frase = generar_frase(s, trigramas, contextos, vocab_dec, n_palabras=8)
    print(f"   [{s:15s}] {' '.join(frase)}")

Tres frases generadas desde 'el':


TypeError: muestrear_siguiente_palabra() missing 1 required positional argument: 'vocab_decodificar'

In [ ]:
def ida_y_vuelta(frase_tokens, vocab_cod, vocab_dec):
    """Codifica una lista de tokens y la vuelve a decodificar."""
    ids = [vocab_cod.get(p, 0) for p in frase_tokens]
    reconstruida = decodificar_secuencia(ids, vocab_dec)
    return ids, reconstruida.split()


def comparar_frases(original, reconstruida):
    """Devuelve un dict con el resultado de la comparacion."""
    return {
        "original": original,
        "reconstruida": reconstruida,
        "longitudes_iguales": len(original) == len(reconstruida),
        "identicas": original == reconstruida,
        "diferencias": [
            (i, a, b)
            for i, (a, b) in enumerate(zip(original, reconstruida))
            if a != b
        ],
    }


# --- Prueba 1: frase generada por el modelo ---
random.seed(42)
frase = generar_frase('el', trigramas, contextos, vocab_dec, n_palabras=10)
ids, reconstruida = ida_y_vuelta(frase, vocab_cod, vocab_dec)
resultado = comparar_frases(frase, reconstruida)

print("Frase generada   :", " ".join(frase))
print("IDs              :", ids)
print("Reconstruida     :", " ".join(reconstruida))
print("Identicas?       ", resultado["identicas"])
print("Diferencias      ", resultado["diferencias"])

# --- Prueba 2: frase con palabra desconocida a proposito ---
print("\n--- Prueba con palabra desconocida ---")
frase_desconocida = ['el', 'futuro', 'cuantico', 'de', 'la', 'ia']
ids2, reconstruida2 = ida_y_vuelta(frase_desconocida, vocab_cod, vocab_dec)
resultado2 = comparar_frases(frase_desconocida, reconstruida2)

print("Original     :", " ".join(frase_desconocida))
print("IDs          :", ids2)
print("Reconstruida :", " ".join(reconstruida2))
print("Identicas?   ", resultado2["identicas"])
print("Diferencias  ", resultado2["diferencias"])

In [ ]:
import json

SEPARADOR = " ||| "


def trigramas_a_json(trigramas):
    """Convierte dict {(w1, w2): n}  ->  dict {'w1 ||| w2': n}."""
    return {f"{w1}{SEPARADOR}{w2}": n for (w1, w2), n in trigramas.items()}


def json_a_trigramas(trigramas_json):
    """Convierte dict {'w1 ||| w2': n}  ->  dict {(w1, w2): n}."""
    return {tuple(clave.split(SEPARADOR)): n for clave, n in trigramas_json.items()}


def guardar_modelo(ruta, vocab_cod, trigramas, contextos):
    """Guarda el modelo completo en un archivo JSON."""
    modelo = {
        "version": 1,
        "vocab_cod": vocab_cod,
        "trigramas": trigramas_a_json(trigramas),
        "contextos": contextos,
    }
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(modelo, f, ensure_ascii=False, indent=2)


def cargar_modelo(ruta):
    """Carga el modelo desde un archivo JSON y reconstruye vocab_dec."""
    with open(ruta, "r", encoding="utf-8") as f:
        modelo = json.load(f)
    vocab_cod = modelo["vocab_cod"]
    trigramas = json_a_trigramas(modelo["trigramas"])
    contextos = modelo["contextos"]
    vocab_dec = construir_vocabulario_decodificar(vocab_cod)
    return vocab_cod, vocab_dec, trigramas, contextos


# --- Prueba de ida y vuelta ---
guardar_modelo("modelo_trigramas.json", vocab_cod, trigramas, contextos)
vc2, vd2, bg2, cx2 = cargar_modelo("modelo_trigramas.json")

print("Vocabulario igual: ", vocab_cod == vc2)
print("trigramas iguales:  ", trigramas  == bg2)
print("Contextos iguales: ", contextos == cx2)
print("Tamano vocab:      ", len(vc2))
print("N trigramas:        ", len(bg2))

# Generamos con el modelo cargado: deberia dar la misma frase
random.seed(42)
frase_cargada = generar_frase('el', bg2, cx2, vd2, n_palabras=10)
print("\nFrase con el modelo cargado:")
print("   ", " ".join(frase_cargada))

# Inspeccion del JSON
print("\n--- Primeras lineas del JSON ---")
with open("modelo_trigramas.json", "r", encoding="utf-8") as f:
    print(f.read()[:600])

In [ ]:
print("=== PRUEBAS UNITARIAS DEL MODELO DE TRIGRAMAS ===\n")

# ---------- Etapa 1 ----------
print("Etapa 1: preparar corpus y vocabulario")

o = preparar_corpus("hola mundo\n\nadios mundo")
assert len(o) == 2, f"Esperaba 2 oraciones, obtuve {len(o)}"
assert o[0] == ['hola', 'mundo']

v = recolectar_vocabulario([['a', 'b', 'a'], ['b', 'c']])
assert v['<UNK>'] == 0
assert v['a'] == 1 and v['b'] == 2 and v['c'] == 3

vd = construir_vocabulario_decodificar(v)
for palabra, idn in v.items():
    assert vd[idn] == palabra

print("   OK - Todas las pruebas de la Etapa 1 pasaron\n")

# ---------- Etapa 2 ----------
print("Etapa 2: contar trigramas y contextos")

bg = contar_trigramas([['a', 'b', 'a', 'b']])
assert bg[('a', 'b')] == 2, f"Esperaba 2, obtuve {bg[('a', 'b')]}"
assert bg[('b', 'a')] == 1

# Los trigramas NO deben cruzar oraciones
bg2 = contar_trigramas([['a', 'b'], ['c', 'd']])
assert ('b', 'c') not in bg2, "Los trigramas no deben cruzar oraciones"

cx = contar_contextos([['a', 'b', 'a', 'b']])
assert cx['a'] == 2 and cx['b'] == 1

print("   OK - Todas las pruebas de la Etapa 2 pasaron\n")

# ---------- Etapa 3 ----------
print("Etapa 3: probabilidades y suavizado de Laplace")

V_prueba = len(vocab_cod)
dist = distribucion_siguiente('el', trigramas, contextos, vocab_dec)
suma = sum(dist.values())
assert abs(suma - 1.0) < 1e-6, f"La distribucion no suma 1, suma {suma}"

# La distribucion debe incluir <UNK>
assert '<UNK>' in dist, "La distribucion debe incluir <UNK>"

p_rara = probabilidad_trigrama('el', 'palabraquenoexiste', trigramas, contextos, V_prueba)
assert p_rara > 0, "Laplace debe evitar el 0"

print("   OK - Todas las pruebas de la Etapa 3 pasaron\n")

# ---------- Etapa 4 ----------
print("Etapa 4: muestreo")

random.seed(0)
for _ in range(50):
    w = muestrear_siguiente_palabra('el', trigramas, contextos, vocab_dec)
    assert w in vocab_cod, f"Palabra fuera del vocabulario: {w}"

# El muestreo nunca debe devolver <UNK>
random.seed(0)
for _ in range(200):
    w = muestrear_siguiente_palabra('el', trigramas, contextos, vocab_dec)
    assert w != '<UNK>', "El muestreo no debe devolver <UNK>"

print("   OK - Todas las pruebas de la Etapa 4 pasaron\n")

# ---------- Etapa 5 ----------
print("Etapa 5: generar frase")

random.seed(42)
fr = generar_frase('el', trigramas, contextos, vocab_dec, n_palabras=10)
assert len(fr) == 10, f"Esperaba 10 palabras, obtuve {len(fr)}"
assert fr[0] == 'el', "La frase debe empezar por la semilla"

print("   OK - Todas las pruebas de la Etapa 5 pasaron\n")

# ---------- Etapa 6 ----------
print("Etapa 6: ida y vuelta")

frase_ok = ['el', 'futuro', 'de', 'la', 'tecnologia']
ids, rec = ida_y_vuelta(frase_ok, vocab_cod, vocab_dec)
assert frase_ok == rec, f"Esperaba {frase_ok}, obtuve {rec}"

frase_mal = ['el', 'cuantico', 'futuro']
ids2, rec2 = ida_y_vuelta(frase_mal, vocab_cod, vocab_dec)
assert ids2[1] == 0, "Palabra desconocida debe ser 0"
assert rec2[1] == '<UNK>', "ID 0 debe decodificarse como <UNK>"

print("   OK - Todas las pruebas de la Etapa 6 pasaron\n")

# ---------- Etapa 7 ----------
print("Etapa 7: guardar y cargar modelo")

guardar_modelo("modelo_test.json", vocab_cod, trigramas, contextos)
vc_t, vd_t, bg_t, cx_t = cargar_modelo("modelo_test.json")
assert vc_t == vocab_cod, "Vocabulario distinto tras cargar"
assert bg_t == trigramas, "Trigramas distintos tras cargar"
assert cx_t == contextos, "Contextos distintos tras cargar"

print("   OK - Todas las pruebas de la Etapa 7 pasaron\n")

print("=== TODAS LAS PRUEBAS PASARON ===")

In [ ]:
# 1) Entrenar
corpus_final = """La Inteligencia Artificial es el futuro de la tecnologia.
Los diccionarios en Python son extremadamente rapidos.
Las tablas hash permiten busquedas en tiempo constante.
La programacion en Python es divertida y poderosa."""

oraciones_f = preparar_corpus(corpus_final)
vocab_cod_f = recolectar_vocabulario(oraciones_f)
vocab_dec_f = construir_vocabulario_decodificar(vocab_cod_f)
trigramas_f = contar_trigramas(oraciones_f)
contextos_f = contar_contextos(oraciones_f)

print(f"Vocabulario: {len(vocab_cod_f)} palabras")
print(f"Trigramas:    {len(trigramas_f)} pares distintos")

# 2) Generar
random.seed(7)
frase_gen = generar_frase('la', trigramas_f, contextos_f, vocab_dec_f, n_palabras=12)
print("\nFrase generada:")
print("   ", " ".join(frase_gen))

# 3) Ida y vuelta
ids_gen, frase_rec = ida_y_vuelta(frase_gen, vocab_cod_f, vocab_dec_f)
print("\nIDs:", ids_gen)
print("Reconstruida:", " ".join(frase_rec))
print("Identicas?", frase_gen == frase_rec)

# 4) Guardar
guardar_modelo("modelo_final.json", vocab_cod_f, trigramas_f, contextos_f)
print("\nModelo guardado en 'modelo_final.json'")

# 5) Cargar desde cero y regenerar
vocab_cod_c, vocab_dec_c, trigramas_c, contextos_c = cargar_modelo("modelo_final.json")
random.seed(7)
frase_gen_c = generar_frase('la', trigramas_c, contextos_c, vocab_dec_c, n_palabras=12)
print("\nFrase desde el modelo cargado:")
print("   ", " ".join(frase_gen_c))
print("\nMisma frase que antes?", frase_gen == frase_gen_c)